# Zelador: assistente de voz de zeladoria urbana (CitizenTech, GS 2027 PLN)
**Autor:** Octavio Augusto Ramalho Brandão Pires, RM 555696

Fluxo: microfone → **STT** (faster-whisper) → **LLM** (Qwen2.5 via Ollama, com RAG e memória) → **adaptação para fala** → **TTS** (Piper) → alto-falante.

**Como usar no Colab:** menu *Ambiente de execução → Alterar tipo → GPU T4*, depois *Executar tudo*. Tudo é gratuito e roda no próprio Colab, sem API paga.

> **Aviso LGPD:** voz é dado pessoal. O áudio do cidadão é apagado logo depois de transcrito e nada é enviado a APIs externas.

**Se algo der erro:** *Ambiente de execução → Reiniciar sessão* e rode de novo a partir da seção 0. Na demonstração (seção 7), se a tela mostrar "Erro", o motivo aparece na própria conversa e no log da célula.

## 0. Obter o projeto
A célula confere a versão do projeto. Se achar uma cópia antiga em `/content`, ela é apagada e a atual é obtida de novo: envie o `.zip` do projeto quando o Colab pedir (ou preencha `REPO_URL` com o link do GitHub depois de enviar esta versão para lá).

In [1]:
import os, shutil, subprocess, sys, zipfile

REPO_URL = "https://github.com/Augusto060908657/PLN-Rodar.git"  # deixe "" se quiser enviar via .zip
VERSAO = "2.0"  # versão do projeto que este notebook espera (src/__init__.py)
RAIZ = "/content" if os.path.isdir("/content") else os.getcwd()

def versao_de(d):
    try:
        with open(os.path.join(d, "src", "__init__.py"), encoding="utf-8") as f:
            return f.read().split('__versao__ = "')[1].split('"')[0]
    except Exception:
        return None

def achar_projeto():
    candidatos = [os.getcwd()]
    for nome in sorted(os.listdir(RAIZ)):
        sub = os.path.join(RAIZ, nome)
        if os.path.isdir(sub):
            candidatos.append(sub)
            for interno in sorted(os.listdir(sub)):
                candidatos.append(os.path.join(sub, interno))
    for d in candidatos:
        if os.path.isdir(d) and versao_de(d) == VERSAO:
            return os.path.abspath(d)
    return None

pasta = achar_projeto()

# Garante que uma pasta clonada do repositório anterior seja substituída pelo novo
if pasta and REPO_URL and os.path.isdir(os.path.join(pasta, ".git")):
    res = subprocess.run(["git", "-C", pasta, "remote", "get-url", "origin"], capture_output=True, text=True)
    if res.stdout.strip() != REPO_URL:
        shutil.rmtree(pasta, ignore_errors=True)
        pasta = None

if pasta is None:
    shutil.rmtree(os.path.join(RAIZ, "projeto"), ignore_errors=True)  # cópia antiga ou incompleta
    if REPO_URL:
        subprocess.run(["git", "clone", REPO_URL, os.path.join(RAIZ, "projeto")], check=True)
    else:
        from google.colab import files
        for nome in files.upload():  # escolha o .zip do projeto
            if nome.endswith(".zip"):
                with zipfile.ZipFile(nome) as zf:
                    zf.extractall(RAIZ)
    pasta = achar_projeto()
elif REPO_URL and os.path.isdir(os.path.join(pasta, ".git")):
    subprocess.run(["git", "-C", pasta, "pull"], check=False)

if pasta is None:
    raise SystemExit("Não achei o projeto na versão " + VERSAO + ". Envie o .zip corrigido (ou suba esta versão ao GitHub).")

os.chdir(pasta)
for m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:  # esquece módulos antigos já importados
    del sys.modules[m]
print("Pasta do projeto:", os.getcwd(), "| versão", versao_de(pasta))

Pasta do projeto: /content/projeto/GS-PLN-Repositorio | versão 2.0


## 1. Instalação (pip, ffmpeg, Ollama)

In [2]:
!pip install -q -r requirements.txt
!pip install -q nvidia-cublas-cu12 nvidia-cudnn-cu12
!apt-get -qq install -y ffmpeg zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

import importlib.metadata as md, subprocess, sys, time, urllib.request

def ollama_ativo():
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2); return True
    except Exception:
        return False

if not ollama_ativo():
    subprocess.Popen(["ollama", "serve"], stdout=open("/tmp/ollama.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(30):
        if ollama_ativo(): break
        time.sleep(2)
print("Ollama no ar" if ollama_ativo() else "Falha ao iniciar o Ollama: " + open("/tmp/ollama.log").read())

for pkg in ("av", "faster-whisper", "ctranslate2", "gradio", "piper-tts"):
    try: print(f"{pkg:15s}", md.version(pkg))
    except md.PackageNotFoundError: print(f"{pkg:15s} NÃO INSTALADO")
if "av" in sys.modules and sys.modules["av"].__version__ != md.version("av"):
    print("ATENÇÃO: o pacote av foi trocado com a sessão aberta. Reinicie a sessão (Ambiente de execução → Reiniciar sessão) e rode de novo a partir da seção 0.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
hf-gradio 0.4.1 requires gradio-client<3.0,>=2.0, but you have gradio-client 1.14.0 which is incompatible.
google-adk 2.7.1 requires starlette<2,>=1.3.1, but you have starlette 0.52.1 which is incompatible.
google-genai 2.12.1 requires pydantic<3.0.0,>=2.12.5, but you have pydantic 2.12.3 which is incompatible.
python-fasthtml 0.14.18 requires starlette>=1.0.1, but you have starlette 0.52.1 which is incompatible.
Ollama no ar
av              18.1.0
faster-whisper  1.2.1
ctranslate2     4.9.0
gradio          5.50.0
piper-tts       1.8.0


In [3]:
# Modelos do LLM (comparação do LLM: 7B x 3B) e voz do TTS
!ollama pull qwen2.5:7b
!ollama pull qwen2.5:3b
from src.tts import baixar_voz
baixar_voz()
import os; print(os.listdir("modelos/piper"))



Baixando pt_BR-faber-medium.onnx ...
Baixando pt_BR-faber-medium.onnx.json ...
['pt_BR-faber-medium.onnx', 'pt_BR-faber-medium.onnx.json']


## 2. Teste rápido de cada módulo

In [4]:
from src.fala import adaptar_para_fala
print(adaptar_para_fala("**Claro!** Ligue 156 ou use o app SP156. Seu protocolo é ZU-20261112-0001, CEP 01301-000. 😊"))

Claro! Ligue um cinco seis ou use o aplicativo S P um cinco seis. Seu protocolo é Z U, dois zero dois seis, um um, um dois, número zero zero zero um. Cép zero um três zero um, zero zero zero.


In [5]:
from src.config import KB_DIR
from src.rag import BaseConhecimento
kb = BaseConhecimento(KB_DIR, 5, 0.12)
print(len(kb.chunks), "trechos na base de conhecimento")
for s, c in kb.recuperar("Como peço o conserto de um buraco?")[:3]: print(round(s, 2), c["texto"][:110])

44 trechos na base de conhecimento
0.39 Buraco na rua (tapa-buraco): O Zelador não sabe o prazo exato do conserto. O prazo depende da vistoria e da fi
0.3 Semáforos e sinalização: O Zelador não sabe o prazo exato do conserto. Acompanhe pelo protocolo no SP156.
0.3 Buraco na rua (tapa-buraco): Buraco em rua ou avenida é pedido pelo serviço de tapa-buraco do SP156. Informe a


In [6]:
from src.pipeline import Assistente
from src.stt import STT; from src.tts import TTS
from src.llm import OllamaChat; from src.dialogo import Dialogo
llm = OllamaChat("qwen2.5:7b"); llm.aquecer()          # carrega o modelo na GPU (a primeira vez demora)
zel = Assistente(STT("faster-whisper", "medium"), Dialogo(llm, kb), TTS(), pasta_audio_saida="/tmp/zel")
r = zel.turno_texto("Oi, quero relatar um buraco enorme na rua Augusta, perto do número quinhentos e vinte.")
print(r["resposta"]); print({k: round(v, 2) for k, v in r.items() if k.endswith("_s")})
from IPython.display import Audio; Audio(r["audio"])

Posso registrar o pedido? É um buraco grande na rua Augusta, próximo ao número 520?
{'stt_s': 0.0, 'rag_s': 0.0, 'llm_s': 75.06, 'fala_s': 0.0, 'tts_s': 0.71, 'total_s': 75.78}


## 3. Avaliação do STT (WER com `jiwer`, 24 áudios, 6 vozes, 6 com ruído) e comparação small × medium

In [7]:
!python -m src.avaliacao_stt --backend faster-whisper --modelos small medium
!python -m src.analise_vies

faster-whisper/small: WER médio (conjunto oficial) = 0.080
faster-whisper/medium: WER médio (conjunto oficial) = 0.050
== por_modelo ==
        audios  wer_medio  wer_mediano  tempo_stt_medio_s
modelo                                                   
medium      24       0.05        0.056              0.605
small       24       0.08        0.065              0.363

== por_condicao ==
condicao  limpo  ruido
modelo                
medium    0.047  0.057
small     0.085  0.063

== por_pessoa ==
modelo   medium  small
pessoa                
Caetano   0.016  0.025
Karen     0.079  0.075
Marcos    0.101  0.209
Octavio   0.029  0.042
Pedace    0.050  0.050
Raio      0.023  0.078

== pessoa_condicao ==
condicao        limpo  ruido
modelo pessoa               
medium Caetano  0.000  0.062
       Karen    0.079  0.077
       Marcos   0.107  0.083
       Octavio  0.000  0.118
       Pedace   0.067  0.000
       Raio     0.030  0.000
small  Caetano  0.033  0.000
       Karen    0.074  0.077
     

## 4. Avaliação do LLM (30 perguntas: 15 do tema, 5 de continuação, 5 fora do escopo, 5 de emergência/limites)

In [8]:
!python -m src.avaliacao_llm --modelos qwen2.5:7b qwen2.5:3b

qwen2.5:7b | tema: 8/15
qwen2.5:7b | continuacao: 4/5
qwen2.5:7b | fora_escopo: 5/5
qwen2.5:7b | limite: 5/5
qwen2.5:7b | total: 22/30  -> /content/projeto/GS-PLN-Repositorio/resultados/llm_qwen2.5_7b.csv
qwen2.5:3b | tema: 10/15
qwen2.5:3b | continuacao: 4/5
qwen2.5:3b | fora_escopo: 5/5
qwen2.5:3b | limite: 4/5
qwen2.5:3b | total: 23/30  -> /content/projeto/GS-PLN-Repositorio/resultados/llm_qwen2.5_3b.csv


## 5. Avaliação do TTS (10 respostas; as notas de naturalidade e clareza vêm de 3 pessoas)

In [9]:
!python -m src.avaliacao_tts
from IPython.display import Audio, display
import glob, os
for f in sorted(glob.glob("resultados/tts_amostras/*.wav")): print(os.path.basename(f)); display(Audio(f))

Amostras geradas em /content/projeto/GS-PLN-Repositorio/resultados/tts_amostras - ouça e preencha a planilha de avaliação (3 pessoas).
T01.wav


T02.wav


T03.wav


T04.wav


T05.wav


T06.wav


T07.wav


T08.wav


T09.wav


T10.wav


## 6. Teste ponta a ponta (10 conversas com 3 ou mais turnos) e latência por etapa

In [10]:
!python -m src.avaliacao_e2e --stt-modelo medium --llm qwen2.5:7b
import pandas as pd
d = pd.read_csv("resultados/e2e_latencia.csv")
print(d[["stt_s", "rag_s", "llm_s", "fala_s", "tts_s", "total_ate_audio_s"]].mean().round(2))

Concluído. Veja resultados/e2e_latencia.csv e resultados/e2e_conversas.txt
stt_s                0.54
rag_s                0.00
llm_s                1.45
fala_s               0.00
tts_s                1.13
total_ate_audio_s    3.12
dtype: float64


## 7. Demonstração com microfone e upload (Gradio). Use esta célula para gravar o vídeo
A primeira execução carrega o STT e o LLM na GPU antes de abrir a interface (1 a 3 minutos). Depois, abra o link `gradio.live`, marque o aceite e use o microfone ou envie um áudio.

In [11]:
import subprocess, time, urllib.request

def ollama_ativo():
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2); return True
    except Exception:
        return False

if not ollama_ativo():
    subprocess.Popen(["ollama", "serve"], stdout=open("/tmp/ollama.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(30):
        if ollama_ativo(): break
        time.sleep(2)
!ollama pull qwen2.5:7b

from src.app_gradio import criar_app
app = criar_app("faster-whisper", "medium", "qwen2.5:7b")     # carrega e aquece STT e LLM
app.launch(share=True, debug=True)


[aquecimento] STT pronto
[aquecimento] LLM pronto
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://df0cac8acc86d96ac2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://df0cac8acc86d96ac2.gradio.live


## 8. Gerar planilhas e relatório com os números reais e baixar tudo

In [12]:
!python -m src.gerar_planilhas
!python -m src.gerar_relatorio
!zip -r -q resultados_colab.zip resultados planilhas relatorio_zelador.pdf
from google.colab import files; files.download("resultados_colab.zip")

Planilhas em /content/projeto/GS-PLN-Repositorio/planilhas
relatorio_zelador.pdf gerado


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>